# nano Beatrix — the frame protocol's second pass on a rented card

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/geolip-nano-beatrix-trainer/blob/main/colab/nano_frame_pass2.ipynb)

**Crafts**: the nano twin pair at their bank of record, the byte-class table beside the aleph address. **Castor** is the rotary softmax body (`castor-both`, 9.2M at 4,096), **Pollux** the alternating body whose hub half keeps a constant decode memory (`pollux-both`, 8.9M). Each craft trains from birth for 12,000 steps on framed rows: frames of generic sizes opened and closed by the byte registry's tags, some nested, a question that can land on any frame, and one row in four a plain document. The record of every run carries the held-out bits inside each frame by kind, size and segment, the fetch by frame size and depth, the plain-text bits beside, and the cost.

**The pass**: two grids at 4,096 on both twins, two seeds each, under the caption register (the first pass, `nano_frame_grid.ipynb`, read every register). The **capacity axis** asks how many records a lookup frame can hold and still be read: 4, 8 and 16 records a frame, with frames up to 1,024 bytes. The **scrutiny set** asks what nesting and conflict cost: frames nested at 0.6, conflicting definitions of a key at 0.3 (the truth is the latest), the question on a nested frame or on a conflicted key. Twenty runs; C2's price line says how long on this card.

**The cells, in order**: C1 pins the trainer and holds the decision block; C2 builds the data, prints each grid's plan, runs 200 steps of the hardest composition on this card for the price line and probes the push path; C3 is the session, one grid after the other, resume-first, with a grid bar and a step bar per running craft, every finished run shipped to [AbstractPhil/alephllm-nano-beatrix-training](https://huggingface.co/AbstractPhil/alephllm-nano-beatrix-training) as it lands; C4 renders the results table.

**Before C1**: put a Hugging Face token with write access to that repo in this notebook's secrets as `HF_TOKEN` and allow the notebook to read it. Tokenless, everything runs and nothing ships. A file named `STOP` in the session folder ends the session between runs; a restarted session resumes where it stopped. The trainer and its tests live in [AbstractEyes/geolip-nano-beatrix-trainer](https://github.com/AbstractEyes/geolip-nano-beatrix-trainer).


In [ ]:
# C1 — pinned installs + THE DECISION BLOCK. RESTART RUNTIME if the version line below changes.
import os, sys
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'   # a progress bar is not allowed to cost a session
%pip install -q "geolip-nano-beatrix-trainer[corpus] @ git+https://github.com/AbstractEyes/geolip-nano-beatrix-trainer@v0.1.18"
import torch
import geolip.nano_beatrix as N
print('geolip.nano_beatrix', N.__version__, '| torch', torch.__version__, '| card', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert tuple(int(x) for x in N.__version__.split('.')) >= (0, 1, 18), (
    'RESTART REQUIRED: the runtime still holds an older package — Runtime > Restart, then rerun from C1')
assert torch.cuda.is_available(), 'no card: Runtime > Change runtime type > GPU'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception:
        return os.environ.get('HF_TOKEN')
HF_TOKEN = _token()
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN            # the runner ships through the environment; the value is never printed

# ============ THE DECISION BLOCK — every setting set from the record, with its source beside it ============
OUT = '/content/out'                                   # the session folder: the corpus, the records, the weights, runner.log
REPO = 'AbstractPhil/alephllm-nano-beatrix-training'  # every finished run ships here (its record and its weights); tokenless = nothing ships
CORPUS_BYTES = 160e6                                   # fineweb-edu text for the bed: 160 MB covers 12,000 steps x 8 x 4,096 four times over
BASE = dict(
    arms='castor-both,pollux-both',   # the twins at the decided bank (table + address, 2026-10-12): the best text on both bodies on two seeds
    seeds='1,2',                      # the two-seed rule
    register='caption',               # the first pass read every register (letters, digits, hex); the second pass holds the register still
    ctx=4096, mb=8, steps=12000,      # the recipe of the first pass: 12,000 steps x 8 rows x 4,096
    frames='dyn', plant=0.75,         # three rows in four framed, one in four a plain document: an all-framed diet made a frame specialist of one
                                      # Castor seed that could not read a bare document (the 2026-10-13 probe); the 4090's night of 10-13 reads the diet
    frame_records=4, nest=0.3, overlap=0.0, ask='any', tags='mode',   # the first pass's frame settings: four records a lookup frame, nesting 0.3,
                                      # no conflicts, the question on any frame, MODE + letter tags; each grid below moves one of them
    log_every=100,                    # the trainer's progress line (step, bits, tok/s, time left) feeds the step bar
)
GRIDS = dict(
    # the in-window capacity axis: how many records a lookup frame holds and still reads; frames up to 1,024 bytes so sixteen records fit
    capacity=dict(BASE, sizes='32,128,256,512,1024', frame_records='4,8,16'),
    # the scrutiny set: nesting at 0.6, conflicting definitions at 0.3 (the truth is the latest), the question on a nested frame or a conflicted key
    scrutiny=dict(BASE, sizes='32,128,256,512', nest=0.6, overlap=0.3, ask='nested,overlap'),
)
SMOKE_STEPS = 200                     # C2's short run on this card for the price line; its push is off
PARALLEL = 1                          # runs at once on the card. ONE: a 9M craft at 8 x 4,096 saturates the RTX PRO 6000's compute alone (863k tok/s);
                                      # four at once ran at ~214k each, their sum one run's rate (2026-10-13): spare VRAM is not spare compute. More than one pays
                                      # only when a single run leaves the card idle, and the proof is the step bars' per-run tok/s summing above one run's rate.
print('decision block:', ' + '.join(f'{k} ({len(g)} keys)' for k, g in GRIDS.items()), '| out', OUT, '| ships to', REPO if HF_TOKEN else 'nothing (tokenless)')


In [ ]:
# C2 (P) — PREFLIGHT: the data, each grid's plan, a short run of the hardest composition on this card (the price line), the push probe. No grid training.
import json, os, time, torch
from geolip.nano_beatrix import bed_corpus, bed_runner

# the data: the corpus and the held-out head from fineweb-edu (tokenless; a few minutes; the lines carry the time left)
if not os.path.exists(f'{OUT}/bed/bed_corpus.bin'):
    bed_corpus.build(f'{OUT}/bed', CORPUS_BYTES)
print('corpus', os.path.getsize(f'{OUT}/bed/bed_corpus.bin') / 1e6, 'MB; held-out head', os.path.exists(f'{OUT}/bed/heldout_fineweb_edu_head2048.jsonl'))

# the plans: what each grid expands to, and what this folder already holds
plans = {}
for name, g in GRIDS.items():
    print('== grid', name)
    plans[name] = bed_runner.run(g, out=OUT, dry=True)

# the price line: the pass's hardest composition (frames to 1,024 bytes, sixteen records a frame) for SMOKE_STEPS on this card
# (the step bar runs; the record stays local under its own short-run name)
smoke = dict(GRIDS['capacity'], arms=BASE['arms'].split(',')[0], seeds='1', frame_records='16', steps=SMOKE_STEPS, log_every=20)
res = bed_runner.run(smoke, out=OUT)
assert res['failed'] is None, 'the short run failed: read the lines above before any grid'
last = [s for s in (bed_runner.parse_step(l) for l in open(f'{OUT}/runner.log', encoding='utf-8')) if s][-1]
per_run = BASE['steps'] * BASE['ctx'] * BASE['mb'] / (last['tok_s'] * 1e3) / 60
todo = sum(p['runs'] - p['done_before'] for p in plans.values())
print(f"this card: {last['tok_s']}k tok/s at {BASE['ctx']} x {BASE['mb']} -> one {BASE['steps']:,}-step run about {per_run:.0f} min alone; "
      f"the {todo} runs left about {per_run * todo / 60:.1f} h one at a time, less with {PARALLEL} at once")

# the push probe (the ship-complete law): a write to the training repo and a read-back BEFORE any training. Tokenless = disarmed.
PUSH_OK = False
if HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    path = 'bed/preflight/push_probe_pass2.json'
    body = json.dumps({'utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'card': torch.cuda.get_device_name(0),
                       'tok_s': last['tok_s'], 'grids': GRIDS}).encode()
    api.upload_file(path_or_fileobj=body, path_in_repo=path, repo_id=REPO, commit_message='nano frame pass 2: push probe')
    PUSH_OK = path in set(api.list_repo_files(REPO))
    print('push probe', 'READ BACK OK' if PUSH_OK else 'NOT VISIBLE after upload', path)
else:
    print('TOKENLESS: no probe written; C3 runs and ships nothing')


In [ ]:
# C3 (T) — THE SESSION: the grids one after the other, PARALLEL runs at once inside each, resume-first (runner_done.txt in OUT); a STOP file
# in OUT starts no new run, lets the running ones finish and skips the grids after; a grid bar over the runs and a step bar per running child fed
# by the trainer's own progress lines; the job-size and gauge lines print through the bars; every finished run ships to REPO in the background
# as it lands: a Hub fault (a 504) retries with a doubling wait and never stops the grid; a record left unshipped is re-shipped at the next
# start and at the end (runner_shipped.txt; the result's 'unshipped' names any that failed). A restart resumes from the records on disk.
# An interrupt loses only the running runs (the bed has no mid-run checkpoint); rerun the cell to continue.
from geolip.nano_beatrix import bed_runner
for name, g in GRIDS.items():
    res = bed_runner.run(g, out=OUT, push=REPO if (HF_TOKEN and PUSH_OK) else '', parallel=PARALLEL)
    print('grid', name, ':', {k: v for k, v in res.items() if k != 'ran'}, '| ran', len(res['ran']))
    if res['stopped']:
        print('STOP seen: the grids after', name, 'are not started; remove the file and rerun the cell to continue')
        break


In [ ]:
# C4 — THE RESULTS: the index over this session's records; the framed rows table carries the text, the bits outside the frames, the data
# frames' body bits by size, the whole value fetched from a frame by its size, the nested whole and the cost. Run any time; the Hub holds the same records.
from IPython.display import Markdown, display
from geolip.nano_beatrix.bed_table import render
display(Markdown(render(f'{OUT}/bed', pairs=True)))


### The other grids

Add, drop or edit an entry of `GRIDS` in C1, rerun C1 (the install is cached), then C3; finished runs are skipped.

| grid | the entry in `GRIDS` |
|---|---|
| the weights source (facts the craft holds in its weights, the question on them) | `weights=dict(BASE, facts='64,256,1024', ask='weights')` — smoked on the 4090, no full run on any card yet: let C2's short run use it first |
| the all-framed diet (task crafts that always see frames) | `plant=1.0` in `BASE`; read the plain-document bits beside the frame gauges, they are the canary |
| the register grid (letters, digits, hex under their own tags) | the first pass: `nano_frame_grid.ipynb` |

A smaller card: `mb=4` at 4,096 (the step count stays; a run takes longer). The session folder is the Colab disk: the Hub copy is the record; a new session rebuilds the corpus in C2 and resumes nothing unless `/content/out/runner_done.txt` is restored.
